
# 贝叶斯分类器实验

## 第一部分

### 实验目的：

掌握贝叶斯分类器算法。

### 实验要求：
不使用sklearn的MultinomialNB，通过自己编程实现朴素贝叶斯算法，并将其应用于课程中讨论的西瓜3.0数据集。

给定一个新的测试样本 (例如, ['乌黑', '硬挺', '沉闷', '清晰', '稍凹', '软粘', 0.500, 0.300])，使用提供的训练数据手动计算样本是好瓜的概率。

选做: 计算'敲声'属性的条件概率时引入一个权重因子（例如乘以2），并说明这样做的原因是基于观察到的模型表现，以及这种改变可能对模型整体性能的影响.

In [2]:
# %load Bayesian.py
import numpy as np
import math

#训练数据集：
dataSet = [
        ['青绿', '蜷缩', '浊响', '清晰', '凹陷', '硬滑', 0.697, 0.460, 1],
        ['乌黑', '蜷缩', '沉闷', '清晰', '凹陷', '硬滑', 0.774, 0.376, 1],
        ['乌黑', '蜷缩', '浊响', '清晰', '凹陷', '硬滑', 0.634, 0.264, 1],
        ['青绿', '蜷缩', '沉闷', '清晰', '凹陷', '硬滑', 0.608, 0.318, 1],
        ['浅白', '蜷缩', '浊响', '清晰', '凹陷', '硬滑', 0.556, 0.215, 1],
        ['青绿', '稍蜷', '浊响', '清晰', '稍凹', '软粘', 0.403, 0.237, 1],
        ['乌黑', '稍蜷', '浊响', '稍糊', '稍凹', '软粘', 0.481, 0.149, 1],
        ['乌黑', '稍蜷', '浊响', '清晰', '稍凹', '硬滑', 0.437, 0.211, 1],
        ['乌黑', '稍蜷', '沉闷', '稍糊', '稍凹', '硬滑', 0.666, 0.091, 0],
        ['青绿', '硬挺', '清脆', '清晰', '平坦', '软粘', 0.243, 0.267, 0],
        ['浅白', '硬挺', '清脆', '模糊', '平坦', '硬滑', 0.245, 0.057, 0],
        ['浅白', '蜷缩', '浊响', '模糊', '平坦', '软粘', 0.343, 0.099, 0],
        ['青绿', '稍蜷', '浊响', '稍糊', '凹陷', '硬滑', 0.639, 0.161, 0],
        ['浅白', '稍蜷', '沉闷', '稍糊', '凹陷', '硬滑', 0.657, 0.198, 0],
        ['乌黑', '稍蜷', '浊响', '清晰', '稍凹', '软粘', 0.360, 0.370, 0],
        ['浅白', '蜷缩', '浊响', '模糊', '平坦', '硬滑', 0.593, 0.042, 0],
        ['青绿', '蜷缩', '沉闷', '稍糊', '稍凹', '硬滑', 0.719, 0.103, 0]
    ]

feature = ['色泽','根蒂','敲声','纹理','脐部','触感','密度','含糖率']

#离散属性取值数目
Num = [3,3,3,3,3,2]

#测试样本1
testData = ['青绿','蜷缩','浊响','清晰','凹陷','硬滑',0.697,0.460]

def Bayesian(trainSet,testData):
    m = len(trainSet)                  #样本数
    goodMelon = []; badMelon = []
    for i in range(m):
        if trainSet[i][-1] == 1:
            goodMelon.append(trainSet[i])
        else:
            badMelon.append(trainSet[i])
            
    p_good = 1.0
    p_bad = 1.0
    
    #计算先验概率
    p_good = p_good*(len(goodMelon)+1)/(m+2)
    p_bad = p_bad*(len(badMelon)+1)/(m+2)    
    
    #计算条件概率-离散
    d1 = len(Num)
    print("########好瓜属性条件概率-离散########")
    for i in range(d1):
        Dc = 0 #不同属性与样本属性相同的个数
        Ni = Num[i]
        for j in range(len(goodMelon)):
            if testData[i] == goodMelon[j][i]:
                Dc = Dc + 1
        p_g = (Dc+1)/(len(goodMelon)+Ni)
        print(testData[i],p_g)
        p_good = p_good*p_g
    
    print("########坏瓜属性条件概率-离散########")
    for i in range(d1):
        Dc = 0 #不同属性与样本属性相同的个数
        Ni = Num[i]
        for j in range(len(badMelon)):
            if testData[i] == badMelon[j][i]:
                Dc = Dc + 1
        p_b = (Dc+1)/(len(badMelon)+Ni)
        print(testData[i],p_b)
        p_bad = p_bad*p_b
    
    #计算条件概率-连续 
    density_sugar_good = [sample[6:8] for sample in goodMelon] #提取连续值
    density_sugar_bad = [sample[6:8] for sample in badMelon] #提取连续值
    ave_good = np.mean(density_sugar_good, axis = 0)
    ave_bad = np.mean(density_sugar_bad, axis = 0)
    std_good = np.std(density_sugar_good, axis = 0, ddof=1) #ddof=1:样本标准偏差
    std_bad = np.std(density_sugar_bad, axis = 0, ddof=1)
    print("########好瓜属性条件概率-连续########")
    for i in range(2):
        p_g = 1/(np.sqrt(2*math.pi)*std_good[i])*math.exp(-(testData[6+i]-ave_good[i])**2/(2*std_good[i]**2))
        print(feature[6+i],p_g) 
        p_good = p_good*p_g
    print("########坏瓜属性条件概率-连续########")
    for i in range(2):
        p_b = 1/(np.sqrt(2*math.pi)*std_good[i])*math.exp(-(testData[6+i]-ave_bad[i])**2/(2*std_bad[i]**2))
        print(feature[6+i],p_b)
        p_bad = p_bad*p_b
    
    #比较好瓜与坏瓜的概率
    if p_good>p_bad:
        print('预测结果为好瓜')
    else:
        print('预测结果为坏瓜')


Bayesian(dataSet,testData)
        

########好瓜属性条件概率-离散########
青绿 0.36363636363636365
蜷缩 0.5454545454545454
浊响 0.6363636363636364
清晰 0.7272727272727273
凹陷 0.5454545454545454
硬滑 0.7
########坏瓜属性条件概率-离散########
青绿 0.3333333333333333
蜷缩 0.3333333333333333
浊响 0.4166666666666667
清晰 0.25
凹陷 0.25
硬滑 0.6363636363636364
########好瓜属性条件概率-连续########
密度 1.9590115494650384
含糖率 0.7880520952044112
########坏瓜属性条件概率-连续########
密度 1.813364315379696
含糖率 0.07072938250447534
预测结果为好瓜


# 第二部分

## 上机内容
1. 探索朴素贝叶斯分类器在文本分类任务中的应用。具体来说，我们使用 fetch_20newsgroups 数据集，该数据集包含来自不同新闻组的约 20,000 篇新闻文章。 
2. 为了简化实验，我们重点关注两个类别：comp.graphics（计算机图形学）和 sci.space（空间科学）。这使我们能够将问题设置为二分类任务，确定给定的新闻文章讨论的是计算机图形学还是空间科学。

## 实验目的：
1. 了解文本分类的基础知识。
2. 掌握朴素贝叶斯算法的应用，包括“make_pipeline”和“MultinomialNB”等。


In [4]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# 加载数据集
categories = ['comp.graphics', 'sci.space']
data = fetch_20newsgroups(subset='all', categories=categories, shuffle=True, random_state=42)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.25, random_state=0)

# 创建一个管道，包含TF-IDF向量化和朴素贝叶斯分类器
model = make_pipeline(TfidfVectorizer(), MultinomialNB())

# 训练模型
model.fit(X_train, y_train)

# 预测测试集
predicted = model.predict(X_test)

# 输出模型的准确率和其他指标
print("Classification report for classifier %s:\n%s\n" % (model, classification_report(y_test, predicted)))
print("Confusion Matrix:\n", confusion_matrix(y_test, predicted))


Classification report for classifier Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer()),
                ('multinomialnb', MultinomialNB())]):
              precision    recall  f1-score   support

           0       0.99      0.92      0.95       250
           1       0.93      0.99      0.96       240

    accuracy                           0.96       490
   macro avg       0.96      0.96      0.96       490
weighted avg       0.96      0.96      0.96       490


Confusion Matrix:
 [[231  19]
 [  3 237]]
